# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nooragab/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

#Setup

In [1]:
%pip install -q duckdb scikit-learn

import duckdb, os
import pandas as pd, numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

build_query = f"""
    WITH march AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS impressions_month,
               SUM(gsc_clicks) AS clicks_month,
               AVG(gsc_avg_position) AS avg_position_month
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    ),
    april AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    tiered AS (
        SELECT m.*, a.impressions_april,
            (a.impressions_april < m.impressions_month)::INT AS is_declining_next_month,
            m.clicks_month / NULLIF(m.impressions_month, 0) AS ctr_month,
            CASE
                WHEN m.avg_position_month <= 3 THEN '1_top_3'
                WHEN m.avg_position_month <= 10 THEN '2_striking_4_10'
                WHEN m.avg_position_month <= 20 THEN '3_page_2'
                ELSE '4_deep_20plus'
            END AS position_tier
        FROM march m
        JOIN april a ON m.content_hash_id = a.content_hash_id
        WHERE m.impressions_month >= 100
    ),
    tier_avg AS (
        SELECT position_tier, AVG(ctr_month) AS tier_avg_ctr FROM tiered GROUP BY position_tier
    )
    SELECT
        t.content_hash_id, t.client_hash_id, t.impressions_month, t.avg_position_month,
        t.position_tier, ROUND(t.ctr_month, 4) AS ctr_month,
        ROUND(ta.tier_avg_ctr, 4) AS tier_avg_ctr,
        ROUND(GREATEST(ta.tier_avg_ctr - t.ctr_month, 0), 4) AS ctr_gap,
        DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_last_update,
        (DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') >= 90)::INT AS is_stale,
        d.word_count, t.is_declining_next_month
    FROM tiered t
    JOIN tier_avg ta ON t.position_tier = ta.position_tier
    JOIN read_parquet('{rel}/dim_content.parquet') d ON t.content_hash_id = d.content_hash_id
"""
data = con.sql(build_query).df()
data["word_count"] = data["word_count"].fillna(data["word_count"].median())

feature_cols = ["impressions_month", "avg_position_month", "ctr_month",
                "tier_avg_ctr", "ctr_gap", "is_stale", "days_since_last_update", "word_count"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data["client_hash_id"]))
train, test = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

X_train = pd.get_dummies(train[feature_cols + ["position_tier"]], columns=["position_tier"])
X_test = pd.get_dummies(test[feature_cols + ["position_tier"]], columns=["position_tier"])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train, y_test = train["is_declining_next_month"], test["is_declining_next_month"]

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(X_train, y_train)

print("Setup complete — data, features, grouped split, and trained model rebuilt from w05.")
print(f"Dataset: {data.shape[0]:,} rows, {data['client_hash_id'].nunique()} clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Setup complete — data, features, grouped split, and trained model rebuilt from w05.
Dataset: 100,893 rows, 43 clients


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1: "The Freshness Multiplier" — Refresh Effect (3.2x health, 57x impressions)

**The claim:** 365+ day pages refreshed within 30 days show a 3.2x health boost (10.7 → 34.5)
and 57x more impressions (71 → 4,039).

**My methodology question:** How were the "refreshed" and "not-refreshed" 365+ groups
selected — was this a random sample of old pages, or were pages chosen for refresh precisely
*because* they already showed signs of recoverable demand? If editors pick which pages to
refresh, the comparison group isn't a fair counterfactual — it compares "pages someone judged
worth saving, after saving them" against "pages nobody bothered with." That selection effect
alone could produce a large gap even if refreshing had a smaller true effect. I'd ask: what
was n for each group, and was there any attempt to match refreshed pages to similar
unrefreshed pages on pre-refresh traffic?

## Finding 2: "What Predicts Growth?" — Logistic Regression, 71% holdout accuracy

**The claim:** a logistic regression trained on portfolio features reaches 71% holdout
accuracy separating growing from declining pages.

**My methodology question:** the methodology section states an "80/20 split," without
specifying whether it's random or grouped by brand. Given the paper covers 57 brands, if
pages from the same brand appear in both train and holdout, the model could be partly
memorizing brand-level patterns rather than a generalizable signal — inflating the 71%
number. I'd also ask whether any input feature's measurement window overlaps the label's
30d-vs-prev-30d window, which would be a soft form of leakage even without a product flag.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Methodology questions asked about:")
print("1. Finding 4 (Freshness Multiplier): selection bias in refresh-vs-not comparison")
print("2. ML Appendix (Growth prediction, 71% accuracy): unspecified split type across 57 brands")

Methodology questions asked about:
1. Finding 4 (Freshness Multiplier): selection bias in refresh-vs-not comparison
2. ML Appendix (Growth prediction, 71% accuracy): unspecified split type across 57 brands


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## My Model Under an Honest Split: Before/After

The paper's growth model uses an unspecified "80/20 split" — exactly the ambiguity I
questioned above. Here I reproduce that risk on my own model: a plain random split (pages
from the same client can land in both train and test) vs. the grouped client-holdout split
I used in w05.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_all = pd.get_dummies(data[feature_cols + ["position_tier"]], columns=["position_tier"])
y_all = data["is_declining_next_month"]

X_tr_random, X_te_random, y_tr_random, y_te_random = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42
)
rf_random = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(X_tr_random, y_tr_random)
p50_random = precision_at_k(rf_random.predict_proba(X_te_random)[:, 1], y_te_random, 50)
p50_grouped = precision_at_k(rf.predict_proba(X_test)[:, 1], y_test, 50)

before_after = pd.DataFrame({
    "split_type": ["random (client-mixed)", "grouped (client-holdout)"],
    "Precision@50": [p50_random, p50_grouped]
})
before_after

,split_type,Precision@50
0,random (client-mixed),0.98
1,grouped (client-holdout),0.72


**This confirms the exact risk I raised about the paper's growth model.** The random
(client-mixed) split scores 0.98 Precision@50 — suspiciously close to perfect — while the
honest grouped (client-holdout) split scores 0.72. That 26-point gap is not real predictive
skill; it's the model partially memorizing client-specific patterns when pages from the same
client appear in both train and test.

This is direct, reproducible evidence that an unspecified "80/20 split" across many clients
(or brands, in the paper's case) can silently inflate a reported accuracy number. My own
w05 result (0.72) is the trustworthy one — it's what the model can actually do on clients
it has never seen before, which is the real-world use case for a ranking queue.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

Same checklist from w03: is each feature calculated before the decision point, does its
window overlap the label window (April), and is it a rebuilt product flag?

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

leakage_audit = pd.DataFrame({
    "feature": feature_cols,
    "measurement_window": ["March"]*5 + ["March (static cutoff)", "March", "static metadata"],
    "overlaps_label_window(April)?": ["No"] * 8,
    "is_product_flag?": ["No"] * 8,
})
print(leakage_audit.to_string(index=False))
print("\nLabel: is_declining_next_month = April impressions < March impressions")
print("Verdict: no feature uses April data. No product decision flags exist in this dataset.")

               feature    measurement_window overlaps_label_window(April)? is_product_flag?
     impressions_month                 March                            No               No
    avg_position_month                 March                            No               No
             ctr_month                 March                            No               No
          tier_avg_ctr                 March                            No               No
               ctr_gap                 March                            No               No
              is_stale March (static cutoff)                            No               No
days_since_last_update                 March                            No               No
            word_count       static metadata                            No               No

Label: is_declining_next_month = April impressions < March impressions
Verdict: no feature uses April data. No product decision flags exist in this dataset.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

**Original claim (from w05):** "the model doesn't just beat the baseline — it revealed that
staleness isn't actually the strongest signal."

**Rewritten:** In this observed sample, the random forest's feature importances suggest
position and impressions carried more predictive weight than staleness. This is a
directional, decision-support finding — it does not prove staleness is irrelevant to content
decay everywhere, only that in this March→April window, on this held-out client split,
position and volume were more informative to this particular model.

**Original claim (from w05):** "the model trades top-of-queue precision for deeper-queue
precision."

**Rewritten:** Precision@20 and Precision@50 moved in different directions when comparing the
baseline rule to the trained models on one held-out split of one month's data. This is an
observed pattern, not a guarantee that will hold across every month or client segment.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Claim audit complete. Two boldest claims from w05 rewritten with:")
print("- 'observed', 'directional', 'decision-support' language")
print("- explicit scope limits (this split, this month, this client sample)")

Claim audit complete. Two boldest claims from w05 rewritten with:
- 'observed', 'directional', 'decision-support' language
- explicit scope limits (this split, this month, this client sample)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.